## Machine Learning model for Used car sales

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Dataset Loading

In [4]:
df = pd.read_csv("cars_24_combined.csv")

### Preprocessing steps

In [5]:
df.head()

,Unnamed: 0,Car Name,Year,Distance,Owner,Fuel,Location,Drive,Type,Price
0,0,Maruti S PRESSO,2022.0,3878,1,PETROL,HR-98,Manual,HatchBack,514000
1,1,Hyundai Xcent,2018.0,32041,1,PETROL,TN-22,Manual,Sedan,674000
2,2,Tata Safari,2021.0,96339,1,DIESEL,TS-08,Automatic,SUV,1952000
3,3,Maruti Vitara Brezza,2019.0,51718,1,DIESEL,WB-24,Manual,SUV,690000
4,4,Tata Tiago,2021.0,19811,1,PETROL,HR-51,Manual,HatchBack,526000


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8015 entries, 0 to 8014
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  8015 non-null   int64  
 1   Car Name    8014 non-null   str    
 2   Year        8014 non-null   float64
 3   Distance    8015 non-null   int64  
 4   Owner       8015 non-null   int64  
 5   Fuel        8015 non-null   str    
 6   Location    7802 non-null   str    
 7   Drive       8015 non-null   str    
 8   Type        8015 non-null   str    
 9   Price       8015 non-null   int64  
dtypes: float64(1), int64(4), str(5)
memory usage: 928.1 KB


In [7]:
df["Car Name"].value_counts()

Car Name
Maruti Swift           593
Hyundai Grand i10      451
Maruti Baleno          430
Hyundai Elite i20      382
Maruti Wagon R 1.0     362
                      ... 
Maruti OMNI E            1
Jeep GRAND CHEROKEE      1
Toyota Fortuner          1
Mahindra MARAZZO         1
Nissan Sunny             1
Name: count, Length: 126, dtype: int64

In [8]:
df.describe()

,Unnamed: 0,Year,Distance,Owner,Price
count,8015.000000,8014.000000,8015.000000,8015.000000,8.015000e+03
mean,4007.000000,2016.995009,52621.411728,1.300187,5.748829e+05
std,2313.875537,2.861454,29182.922728,0.510893,2.651049e+05
min,0.000000,2010.000000,0.000000,1.000000,1.190000e+05
25%,2003.500000,2015.000000,30730.000000,1.000000,3.930000e+05
50%,4007.000000,2017.000000,50359.000000,1.000000,5.350000e+05
75%,6010.500000,2019.000000,71762.000000,2.000000,6.980000e+05
max,8014.000000,2023.000000,971212.000000,4.000000,3.300000e+06


In [9]:
df.shape

(8015, 10)

#### Checking Duplicates


In [10]:
df[df.duplicated()]

,Unnamed: 0,Car Name,Year,Distance,Owner,Fuel,Location,Drive,Type,Price


#### Handling Null values

In [11]:
df.isnull().sum()

Unnamed: 0      0
Car Name        1
Year            1
Distance        0
Owner           0
Fuel            0
Location      213
Drive           0
Type            0
Price           0
dtype: int64

In [12]:
df.dropna(subset=["Car Name", "Year"], inplace=True)

df["Location"] = df["Location"].fillna("Unknown")

In [13]:
df.drop("Unnamed: 0", axis=1, inplace=True)

In [14]:
df.corr(numeric_only=True)["Price"].sort_values(ascending=False)

Price       1.000000
Year        0.501978
Owner      -0.149838
Distance   -0.198750
Name: Price, dtype: float64

In [15]:
df.columns

Index(['Car Name', 'Year', 'Distance', 'Owner', 'Fuel', 'Location', 'Drive',
       'Type', 'Price'],
      dtype='str')

In [16]:
df.isnull().sum()

Car Name    0
Year        0
Distance    0
Owner       0
Fuel        0
Location    0
Drive       0
Type        0
Price       0
dtype: int64

### Splitting features and target

In [17]:
X = df.drop("Price", axis=1)
Y = df["Price"]


In [18]:
X.head()

,Car Name,Year,Distance,Owner,Fuel,Location,Drive,Type
0,Maruti S PRESSO,2022.0,3878,1,PETROL,HR-98,Manual,HatchBack
1,Hyundai Xcent,2018.0,32041,1,PETROL,TN-22,Manual,Sedan
2,Tata Safari,2021.0,96339,1,DIESEL,TS-08,Automatic,SUV
3,Maruti Vitara Brezza,2019.0,51718,1,DIESEL,WB-24,Manual,SUV
4,Tata Tiago,2021.0,19811,1,PETROL,HR-51,Manual,HatchBack


### Encoding

In [19]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
cat_cols = ["Car Name","Fuel","Location","Drive","Type"]
ct = ColumnTransformer(
    [(
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ),
        cat_cols
    )],
    remainder="passthrough"
)

X = ct.fit_transform(X)

X = pd.DataFrame(
    X,
    columns=ct.get_feature_names_out()
)


### Train-test split

In [20]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

### Scaling

In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### MODEL PREDICTION

#### Linear Regression

In [22]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train,Y_train)


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [23]:
y_pred = lr.predict(X_test)

In [24]:
from sklearn.metrics import r2_score,mean_absolute_error,root_mean_squared_error,mean_squared_error
print("====Linear Regression====")
print("r2 score",r2_score(Y_test,y_pred))
print("MAE",mean_absolute_error(Y_test,y_pred))
print(root_mean_squared_error(Y_test,y_pred))

====Linear Regression====
r2 score 0.7681252741497795
MAE 75965.01997570558
130746.95769847061


In [25]:
print(type(Y_test))

<class 'pandas.Series'>


### KNN

In [26]:
from sklearn.neighbors import KNeighborsRegressor
knn = KNeighborsRegressor()
knn.fit(X_train,Y_train)
y_pred = knn.predict(X_test)
print("====KNN====")
print("r2 score",r2_score(Y_test,y_pred))
print("MAE",mean_absolute_error(Y_test,y_pred))
mae_percent = (mean_absolute_error(Y_test, y_pred) / Y_test.mean()) * 100
print(f"MAE % = {mae_percent:.2f}%")

====KNN====
r2 score 0.5063736749628858
MAE 125046.35995009358
MAE % = 21.68%


In [27]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=500,
    max_depth=None,
    )
rf.fit(X_train,Y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [28]:
y_pred = rf.predict(X_test)

In [29]:
print("RANDOM FOREST")
print("r2 score",r2_score(Y_test,y_pred)*100)
print("MAE",mean_absolute_error(Y_test,y_pred))

RANDOM FOREST
r2 score 78.40333301127394
MAE 73139.58721147847


### DECISION TREE

In [30]:
from sklearn.tree import DecisionTreeRegressor
dt = DecisionTreeRegressor()
dt.fit(X_train,Y_train)
y_pred = dt.predict(X_test)
print("DECISION TREE")
print("r2 score",r2_score(Y_test,y_pred)*100)
print("MAE",mean_absolute_error(Y_test,y_pred))

DECISION TREE
r2 score 68.47694870478023
MAE 90013.56830941983


### ADABOOST 

In [31]:
from sklearn.ensemble import AdaBoostRegressor
ab = AdaBoostRegressor()
ab.fit(X_train,Y_train)
y_pred = ab.predict(X_test)
print("====ADABOOST====")
print("r2 score",r2_score(Y_test,y_pred))
print("MAE",mean_absolute_error(Y_test,y_pred))

====ADABOOST====
r2 score 0.37275423188848267
MAE 171341.3184016322


### GRADIENT BOOST

In [32]:
from sklearn.ensemble import GradientBoostingRegressor
gb = GradientBoostingRegressor()
gb.fit(X_train,Y_train)
y_pred = gb.predict(X_test)
print("====GRADIENT BOOST====")
print("r2 score",r2_score(Y_test,y_pred))
print("MAE",mean_absolute_error(Y_test,y_pred))

====GRADIENT BOOST====
r2 score 0.7267150802677238
MAE 90702.82928812163


### XGBOOST

In [33]:
from xgboost import XGBRegressor
xg = XGBRegressor()
xg.fit(X_train,Y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_met

In [34]:
y_pred = xg.predict(X_test)

In [35]:
print("XGBOOST")
print("r2 score",r2_score(Y_test,y_pred)*100)
print("MAE",mean_absolute_error(Y_test,y_pred))

XGBOOST
r2 score 80.02620339393616
MAE 67682.09375


### XGBOOST THROUGH GRIDSEARCHCV

In [36]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42
)
param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [4, 6],
    "learning_rate": [0.05, 0.1]
}
grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,    
)
grid.fit(X_train, Y_train)
print("Best Parameters:")
print(grid.best_params_)
print("\nBest CV R²:")
print(grid.best_score_)

Best Parameters:
{'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 500}

Best CV R²:
0.8409339904785156


In [37]:
best_xgb = grid.best_estimator_

y_pred = best_xgb.predict(X_test)

In [38]:
print("====XGB through GridSearchCV====")
print("r2 score",r2_score(Y_test,y_pred))
print("MAE",mean_absolute_error(Y_test,y_pred))
print("RMSE",root_mean_squared_error(Y_test,y_pred))


====XGB through GridSearchCV====
r2 score 0.8166325092315674
MAE 64801.81640625
RMSE 116269.5625


In [39]:
import joblib
joblib.dump(best_xgb,"model.pkl")
joblib.dump(scaler, "scaler.pkl")  
joblib.dump(ct, "transformer.pkl")

['transformer.pkl']